# `mini_transformer.ipynb` 章节模板

## 先说明三个来自你贴出的源码的设计约束

这三点会影响章节安排：

1. **Mini 模型的参数名要和 staff 完全一致**，`load_state_dict(strict=True)` 才能直接通过。需要对齐的名字有：`token_embeddings.weight`、`layers.{i}.attn.{q,k,v}_proj.weight`、`attn.output_proj.weight`、`ffn.{w1,w2,w3}.weight`、`ln1.weight`、`ln2.weight`、`ln_final.weight`、`lm_head.weight`。RoPE 的缓存是 `persistent=False`，不在 state_dict 里，不用管。
2. **RoPE 的实现有一个值得单独讲的细节**：输入用 `(half_d xy)` 拆成相邻两两配对（交错布局），输出却是 `concat(x1_rot, x2_rot)`（前后半布局）。因为 Q 和 K 经过同一个维度置换，点积不变，所以注意力分数仍然正确。你可以用"教科书写法"，只要 logits 一致就行，但要在 notebook 里解释清楚这个等价性。
3. **`lm_head` 与 embedding 不绑定**（绑定的那行被注释掉了），所以参数量里是 2·V·d，和 Step 2 的公式一致。

## 总体结构

每个模块（第 2 到 8 章）统一用同一个四段式：**数学 → 实现 → 对拍 staff → 代价与 kernel 归属**。得到 Part 1 需要的"代码到 kernel 对照表"。

```
0. 目标、约定与环境
1. 记号与形状约定
2. Embedding
3. Linear
4. RMSNorm
5. RoPE
6. 因果多头自注意力
7. SwiGLU 前馈
8. Transformer Block（pre-norm 残差）
9. 整体拼装：BasicsTransformerLM
10. 与 staff 实现的整体验证
11. 代价核算（FLOPs / 显存 / 访存）
12. 代码 → kernel 对照表与 NVTX 标注点
13. 附录：推导与自测题
```


## 各章内容

### 0. 目标、约定与环境
- 本 notebook 的目标：理解 Part 1 要测的对象，产出 `cs336_systems/mini_model.py`
- 验收标准：`load_state_dict` 通过，logits `allclose`，梯度也 `allclose`
- 环境：用 `uv run` 启动 Jupyter（需要 `uv add --dev ipykernel jupyter`），固定随机种子，统一 fp32 且 TF32 关

### 1. 记号与形状约定
#### 符号表

| 符号 | 含义 | 代码里的名字 | 作业取值 / 关系 | 注解 |
|---|---|---|---|---|
| B | batch size | 输入 `x` 的第 0 维 | 作业 benchmark 固定为 4 | 一次并行处理的序列条数 |
| T | 序列长度（context length） | `context_length`、`sequence_length` | 128 / 256 / 512 / 1024 | 注意力的代价按 T² 增长 |
| N | 一个 batch 的 token 总数 | 无 | N = B·T | 线性层的 FLOPs 只和 N 成正比，和 B、T 的拆分无关 |
| d | 模型宽度 | `d_model` | small 768、medium 1024、large 1280、xl 1600、2.7B 2560 | 残差流的通道数，几乎所有张量的最后一维 |
| h | 注意力头数 | `num_heads` | 12 / 16 / 20 / 25 / 32 | 必须整除 d |
| d_k | 每个头的维度 | `d_k`（`d_v` 与之相等） | d_k = d / h，前四档都是 64，2.7B 为 80 | h·d_k = d，拆头不增加参数量 |
| d_ff | FFN 的内层宽度 | `d_ff` | d_ff = 4d（3072 / 4096 / 5120 / 6400 / 10240） | SwiGLU 有 3 个 d×d_ff 的矩阵 |
| V | 词表大小 | `vocab_size` | 10000 | 决定 embedding 与 lm_head 的大小，也决定 logits 的显存 |
| L | Transformer 层数 | `num_layers` | 12 / 24 / 36 / 48 / 32 | 深度，参数与 FLOPs 都按 L 线性增长 |

补充约定：
- **与 Step 2 的区别**：Step 2 的公式里用 `T = B·ctx` 表示 token 总数。notebook 里 T 已经表示序列长度，所以**token 总数改用 N = B·T**，Step 2 的公式里把 T 换成 N 即可，不要混用。
- 张量形状里用 `...` 表示任意前导维度（与 staff 源码的 jaxtyping 注解一致）。
- 作业不绑定 embedding 与 lm_head，二者各占 V·d 个参数。
- 全模型的张量形状流：`(B,T)` → `(B,T,d)` → `(B,h,T,d_k)` → `(B,T,V)`
- 约定一次，后面每章的形状注释都用这套

```python
"I love deep learning"          ← 原始文本
   ↓ ① 分词器切分
["I", " love", " deep", " learning"]   ← tokens（子词/字符/单词，取决于分词器）
   ↓ ② 查词表：每个 token 字符串 → 一个固定整数
[4, 5, 6, 7]                    ← token_ids
   ↓ ③ Embedding 查表
(5, d_model) 浮点矩阵            ← 进入 Transformer
   ...
   ↓ ④ 模型末端的 LM 头：d_model → vocab_size 个分数
softmax / 采样 → 得到一个 id
   ↓ ⑤ 反查词表 id → token 字符串 → 拼回文本
```

In [2]:
from __future__ import annotations

import json
import logging
import math
import os
import warnings

import einx
import torch
import torch.nn as nn
from einops import einsum, rearrange
from jaxtyping import Bool, Float, Int
from torch import Tensor


### Linear


### 2. Embedding

**2.1 数学**
- 设权重 $W\in\mathbb{R}^{V\times d}$，第 $i$ 行是词 $i$ 的向量。对 id $x$：$E(x)=W_{x,:}$。
- 形状规则：ids 为任意形状 `(...)`，输出为 `(..., d)`。
- 与 one-hot 的关系：令 $O\in\{0,1\}^{N\times V}$，$O_{n,x_n}=1$，则 $Y=OW$。查表就是这个矩阵乘的结果，但**不真的构造 $O$**。
- 代价对比：one-hot 矩阵乘需要 $2NVd$ FLOPs（与 lm_head 同量级），查表是 0 FLOPs，只读 $N\cdot d$ 个元素。

**2.2 可微：梯度是 scatter-add**
- 前向是 gather，反向是 scatter-add：
  $$\frac{\partial L}{\partial W_{i,:}}=\sum_{n:\,x_n=i}\frac{\partial L}{\partial Y_{n,:}}$$
- 三个结论：
  1. 本 batch 没出现的词，对应行的梯度为 0；
  2. 重复出现的词，梯度累加（演示用的 `[0,3,2,2]` 里，词 2 的梯度就是两份之和）；
  3. id 是整数，自身没有梯度。
- 系统视角：梯度张量仍是**稠密的 $V\times d$**（大部分行为 0），AdamW 也会更新所有行，这部分显存开销不随 batch 变化。

**2.3 实现与初始化**
- 实现：`self.weight[token_ids, :]`（高级索引），对照 `F.embedding`。
- 初始化：截断正态，std=1，截断区间 $[-3,3]$（作业规定）。
- 注意：原论文在 embedding 之后会乘 $\sqrt{d_{model}}$，staff 源码**没有乘**，代码注释里已说明；本 notebook 也不乘。

**2.4 对拍**
- 演示单元（代码，输入与输出分开展示）：
  1. $V=5,\ d=4$ 的小权重，`weight[2]`、`weight[[0,3,2,2]]`；
  2. ids 形状 `(2,3)` → 输出 `(2,3,4)`；
  3. 三种实现的等价性：手写查表、`F.embedding`、one-hot 矩阵乘。
- 断言：
  - 手写查表与 `F.embedding`：`torch.equal`；
  - one-hot 矩阵乘：`torch.allclose`；
  - 与 staff 的 `Embedding`，同权重：`torch.equal`。
- 边界：越界 id 在 CPU 上报 `IndexError`，在 GPU 上会触发 device-side assert，调试时先在 CPU 上复现。

**2.5 代价与 kernel 归属**
- 参数量：$V\cdot d$；FLOPs：0；访存：读写各约 $N\cdot d$ 个元素，量级很小。
- 预期 kernel：前向是 gather / index 类；反向是 scatter-add（`index_put` 或 embedding backward 类）。具体名字以 nsys 为准，回填到第 12 章对照表。
- 预期占比：对训练步时间影响很小，但它的稠密梯度和 Adam 状态占用 $V\cdot d$ 的显存。



In [3]:
class Embedding(nn.Module):
    def __init__(self, vocab_size: int, d_model: int):
        super.__init__()
        std = 1.0
        self.weight = nn.Parameter(
            nn.init.trunc_normal_(torch.empty(vocab_size, d_model), std=std, a=-3 * std, b=3 * std), requires_grad=True
            )
    def forward(self,token_ids:Int[Tensor,"..."])->Float[Tensor,"..., d_model"]:
        return self.weight[token_ids,:]


### 3. Linear
- 数学：`y = x Wᵀ`，权重形状为 `(d_out, d_in)` 的原因
- 初始化推导：std = √(2/(d_in+d_out))（Xavier 形式），截断在 ±3σ
- 实现：先写 `x @ W.T`，再解释 staff 用 einsum 的写法，二者等价
- 代价：2·T·d_in·d_out FLOPs，是计算密集型
- 与 Part 1 的关联：einsum 在底层是否额外产生 permute 或 copy，留给 nsys 验证

In [4]:
class Linear(nn.Module):
    def __init__(self,d_in:int,d_out:int):
        # d_in -> d_out
        # initialized with truncated normal fan-in fan-out.
        super.__init__()
        std = math.sqrt(2 / (d_in + d_out))
        self.weight = nn.Parameter(
            nn.init.trunc_normal_(torch.empty(d_out, d_in), std=std, a=-3 * std, b=3 * std), requires_grad=True
        )

    def farward(self,x:Float[Tensor,"... d_in"])->Float[Tensor,"... d_out"]:
        return einsum(x,self.weight,".. d_in, d_out d_in-> ... d_out")

### 4. RMSNorm

#### 4.1 数学：定义

符号：对每个 token 的 $d$ 维向量 $x\in\mathbb{R}^d$ 独立归一化（$d$ 为隐藏维，不是 token 总数 $N=B\cdot T$）。

$$
r(x)=\Big(\tfrac1d\textstyle\sum_{j=1}^d x_j^2+\epsilon\Big)^{-1/2},\qquad
\hat x = r\cdot x,\qquad
y = w\odot \hat x+b
$$

- $w\in\mathbb{R}^d$ 是可学习的逐通道缩放，初始化为 1。作业（staff 实现）**没有 bias**，$\epsilon=10^{-5}$，且 $\epsilon$ 在根号**里面**。
- 含义：只做**重缩放**，让每个向量的均方根变为 1：$\frac1d\sum\hat x_j^2=1$（$\epsilon\to0$ 时）。不做中心化。

**与 LayerNorm 的对比**

$$
\mu=\tfrac1d\textstyle\sum x_j,\quad \sigma^2=\tfrac1d\textstyle\sum (x_j-\mu)^2,\quad
\hat x=\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}},\quad y=w\odot\hat x+b
$$

| | LayerNorm | RMSNorm |
|---|---|---|
| 统计量 | 均值 $\mu$ 和方差 $\sigma^2$ | 只有 $\frac1d\sum x^2$ |
| 中心化 | 有（减 $\mu$） | 无 |
| 可学习参数 | $w$ 和 $b$ | 只有 $w$ |
| 遍历次数 | 可用 $\sum x,\sum x^2$ 一次遍历算出，**不一定多一次扫描** | 一次 |

结论：RMSNorm 省的是一个统计量、一步减法和 $b$ 的参数与梯度，**不是**扫描次数。在访存受限的情况下，两者的性能差异主要看实现是否融合。

#### 4.2 数学：反向传播

记 $g=\delta_y\odot w$（对 $\hat x$ 的梯度），$\delta_y=\partial L/\partial y$。

**推导**（逐分量）：$\hat x_j=r\,x_j$，且 $\partial r/\partial x_i=-\,r^3x_i/d$，所以

$$
\frac{\partial L}{\partial x_i}=\sum_j g_j\frac{\partial \hat x_j}{\partial x_i}
= r\,g_i-\frac{r^3x_i}{d}\sum_j g_jx_j .
$$

利用 $r^3x_ix_j=r\,\hat x_i\hat x_j$，得到

$$
\boxed{\delta_x=r\big(g-c_1\,\hat x\big),\qquad c_1=\tfrac1d\textstyle\sum_j g_j\hat x_j}
$$

参数梯度（对所有 token 求和，即对 batch 维与序列维归约）：

$$
\delta_w=\sum_{b,t}\delta_y\odot\hat x
$$

**LayerNorm 的对应形式**（用于对比）：

$$
\delta_x=r\big(g-c_2-c_1\,\hat x\big),\quad c_1=\tfrac1d\textstyle\sum_j g_j\hat x_j,\quad c_2=\tfrac1d\textstyle\sum_j g_j,
$$
$$
\delta_w=\sum\delta_y\odot\hat x,\qquad \delta_b=\sum\delta_y
$$

多出的 $-c_2$ 来自中心化，它把梯度中"沿全 1 方向"的分量去掉。

**几何理解**
- RMSNorm 的 $\delta_x$ 是把 $g$ 中沿 $\hat x$ 方向的分量去掉后再乘 $r$。
- 原因是尺度不变性：$\hat x(\alpha x)=\hat x(x)$（$\epsilon\to0$），所以梯度必须与 $x$ 正交，即 $\sum_i x_i\,\delta_{x,i}=0$。这是一个可以数值验证的性质。

**系统视角**
- 反向需要保存 $x$（或 $\hat x$）和每行的 $r$（$N$ 个标量）。
- 反向有两类归约：沿 $d$ 的行内归约（$c_1$），和沿 $N$ 的跨行归约（$\delta_w$）。
- 全是逐元素与归约，没有 matmul，属于访存密集。

#### 4.3 为什么先升到 fp32 再算

| 数据类型 | 风险 |
|---|---|
| fp16 | 范围小（最大约 65504），$|x|>256$ 时 $x^2$ 就**溢出** |
| bf16 | 范围同 fp32，不溢出，但尾数只有 8 位；对 $d$ 项求和时**累加误差大**，$r$ 的精度差 |

做法：`x.to(float32)` 之后做平方、均值、rsqrt，最后 `.to(in_dtype)`（staff 源码就是这样）。这样不管外层是否 autocast，归一化的统计量始终在 fp32 里计算。

#### 4.4 实现

- 先写与定义一一对应的版本（升精度、平方、均值、`rsqrt`、乘 $r$、乘 $w$、转回原类型）。
- 参数名必须是 `weight`（形状 `(d,)`），才能被 `load_state_dict` 加载。
- 再写一个**手写反向**版本（`torch.autograd.Function`），用于验证 4.2 的推导。

#### 4.5 对拍

| 检查 | 方法 | 预期 |
|---|---|---|
| 前向与 staff 一致 | 同权重同输入，fp32 | `torch.equal` 或误差 < 1e-6 |
| 与 `torch.nn.RMSNorm` 一致 | 构造时显式 `eps=1e-5`（其默认值不同） | `allclose` |
| 手写反向与 autograd 一致 | **float64**，随机输入，比较 $\delta_x$ 和 $\delta_w$ | `allclose` |
| 尺度不变性 | $\text{RMSNorm}(\alpha x)$ 对比 $\text{RMSNorm}(x)$，$\alpha=3$ | 近似相等（$\epsilon$ 影响很小） |
| 梯度与 $x$ 正交 | 计算 $\sum_i x_i\delta_{x,i}$ | 接近 0 |
| LayerNorm 对照 | 手写 LayerNorm 反向，验证 $-c_2$ 项 | 与 autograd 一致 |

#### 4.6 与 Part 1 的关联：混合精度（3.5）

这一章直接对应 3.5 里"哪些算子在 autocast 下的 dtype"的问题。不要凭推测填表，**用 forward hook 打印实测**：

| 对象 | 验证方法 | 要观察的点 |
|---|---|---|
| RMSNorm 的输入 dtype | hook 里打印 `input[0].dtype` | 残差流是 fp32 还是 bf16 |
| RMSNorm 的输出 dtype | hook 里打印 `output.dtype` | 是否等于输入 dtype（代码里 `.to(in_dtype)`） |
| `weight` 的 dtype | `self.weight.dtype` | 参数始终是 fp32 |
| 归一化内部的统计量 | 看源码 | 显式升到 fp32，与 autocast 无关 |

#### 4.7 代价与 kernel 归属

**FLOPs**：每个元素约几次乘加，总量约 $O(N\cdot d)$，与同层 matmul 的 $O(N\cdot d^2)$ 相比小一个 $d$ 的量级，可忽略。

**访存**：这才是重点。staff 的实现是未融合的 PyTorch 小算子，大致是：

| 步骤 | kernel 类型 |
|---|---|
| `x.to(float32)` | 类型转换（输入已是 fp32 时没有） |
| `x.pow(2)` | elementwise |
| `.mean(-1)` | reduction |
| `+ eps`、`rsqrt` | elementwise（很小） |
| `x * rms` | elementwise |
| `self.weight * x` | elementwise |
| `.to(in_dtype)` | 类型转换（同类型时没有） |

约 6 到 8 个 kernel，每个都要完整读写一遍 $N\times d$ 的张量，而融合后的 kernel 只需读一次写一次。

**量级估算**（small，B=4，ctx=512，fp32）：
- 张量大小：$2048\times768\times4$ B ≈ 6.3 MB
- 每个 elementwise kernel 的读写：约 12.6 MB，按 769 GB/s 约 16 µs
- 约 8 个 kernel → 约 0.13 ms；每次前向有 $2\times12+1=25$ 次 RMSNorm → 约 3 ms
- 对比 small 的 forward 预期 matmul 时间约 16 ms，**约增加 20%**，且 kernel 小，发射开销也不可忽略

这是一个可以在 nsys 里验证的预测，也是 3.7（`torch.compile`）的预期收益来源：编译会把这些小算子融合。

**回填第 12 章对照表**：RMSNorm 的 kernel 名、调用次数、占前向时间的百分比。

In [5]:
class RMSNorm(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        eps: float = 1e-5,
        device=None,
        ):
        self.weight = nn.Parameter(torch.ones(hidden_size, device=device))
        self.eps = eps
    
    def forward(self,x):
        # no bias
        in_dtype = x.dtype()

        x = x.to(torch.float32)
        rms = torch.rsqrt(x.pow(2).mean(-1,deepdim=true)+self.eps)
        x = x*rms
        return (self.weight*x).to(in_dtype)

### 5. RoPE（旋转位置编码）

**符号**：RoPE 作用在 **Q、K 的每个头**上（拆头之后），维度为 $d_k$（须为偶数）。
基数 $\Theta$ = `rope_theta` = 10000；第 $j$ 对的频率 $\theta_j$；位置 $m,n\in\{0,\dots,T-1\}$。
V 和 embedding 都**不**做 RoPE。

#### 5.1 目标

希望给 $q$（位置 $m$）和 $k$（位置 $n$）各施加一个与位置有关的变换 $f$，使注意力分数只依赖**相对位置**：

$$\langle f(q,m),\,f(k,n)\rangle = g(q,k,m-n)$$

位置信息直接进入 $QK^\top$，不需要额外加到 embedding 上，也没有可学习参数。

#### 5.2 构造与证明

**二维情形**：把向量相邻两个分量 $(x_{2j},x_{2j+1})$ 看作平面上的点，逆时针旋转角度 $m\theta_j$：

$$
\begin{bmatrix}y_{2j}\\y_{2j+1}\end{bmatrix}
=R(m\theta_j)\begin{bmatrix}x_{2j}\\x_{2j+1}\end{bmatrix},\quad
R(\alpha)=\begin{bmatrix}\cos\alpha&-\sin\alpha\\ \sin\alpha&\cos\alpha\end{bmatrix}
$$

整个 $d_k$ 维向量被分成 $d_k/2$ 对，每对用自己的频率 $\theta_j$，等价于一个块对角旋转矩阵。

**复数形式**：令 $z_j=x_{2j}+\mathrm{i}\,x_{2j+1}$，则 $y_j=z_j\,e^{\mathrm{i}m\theta_j}$。

**相对位置性质的证明**（复数版，最简洁）：二维点积等于 $\mathrm{Re}(a\bar b)$，所以

$$
\langle f(q,m),f(k,n)\rangle=\mathrm{Re}\!\big(q e^{\mathrm{i}m\theta}\cdot\overline{k e^{\mathrm{i}n\theta}}\big)
=\mathrm{Re}\!\big(q\bar k\,e^{\mathrm{i}(m-n)\theta}\big)
$$

只依赖 $m-n$。矩阵版同理：$R(m\theta)^\top R(n\theta)=R((n-m)\theta)$，所以
$q^\top R(m\theta)^\top R(n\theta)\,k=q^\top R((n-m)\theta)\,k$。
对 $d_k/2$ 对求和，整体内积仍只依赖 $m-n$。

**两个副产品**：
- 旋转是正交变换，**不改变 $q,k$ 的范数**；
- 位置越近，同一频率下的相位差越小。

#### 5.3 频率设计

$$\theta_j=\Theta^{-2j/d_k},\qquad j=0,\dots,\tfrac{d_k}{2}-1,\qquad \text{波长 }\lambda_j=\tfrac{2\pi}{\theta_j}$$

以 $d_k=64$、$\Theta=10^4$ 为例：

| 通道对 | $\theta_j$ (rad/token) | 波长 (tokens) | 含义 |
|---|---|---|---|
| $j=0$ | 1 | 约 6.3 | 转得最快，分辨相邻位置 |
| $j=15$ | 约 $10^{-2}$ | 约 630 | 中等尺度 |
| $j=31$ | 约 $1.3\times10^{-4}$ | 约 47000 | 几乎不转 |

- **为什么是几何级数**：波长从几个 token 到几万 token 在**对数轴上均匀分布**，让不同通道负责不同距离尺度。这个选择继承自 Transformer 原始的正弦位置编码，是经验设计，不是唯一解；RoFormer 论文还从"内积随相对距离衰减"的角度给出了论证（可自行查阅）。
- **一个值得注意的事实**：训练 ctx 只有 128 到 1024 时，最后几对通道的累计转角很小（$j=31$ 在 $m=1024$ 处只转约 0.14 rad），它们近似当作"与位置无关"的通道。
- **$\Theta$ 的作用**：增大 $\Theta$ 会拉长最大波长，是长上下文外推常用的调节旋钮。

#### 5.4 可微：反向就是逆旋转

RoPE 没有参数，$R$ 是正交矩阵，所以：

$$\delta_x = R(m\theta_j)^\top\delta_y = R(-m\theta_j)\,\delta_y$$

即把 $\sin$ 的符号取反，其余形式与前向相同，不需要保存任何激活（cos/sin 表是常量）。

#### 5.5 算法：不要真的构造旋转矩阵

块对角矩阵乘法每个向量需要约 $2d_k^2$ FLOPs，没有必要。逐对用逐元素公式：

$$y_1=\cos\cdot x_1-\sin\cdot x_2,\qquad y_2=\sin\cdot x_1+\cos\cdot x_2$$

每对 4 次乘法、2 次加减，约每个元素 3 FLOPs。

**cos/sin 缓存**
- 形状 `(2, context_length, d_k/2)`：第 0 维是 cos 和 sin，第 1 维是位置，第 2 维是频率对。
- 构造：`freqs = Θ ** (-arange(0, d_k, 2) / d_k)`，再与 `arange(context_length)` 做外积，取 cos、sin 后 stack。
- 在 staff 源码里以 `register_buffer(..., persistent=False)` 注册，**不进入 state_dict**，且全部层共用同一份。
- 缓存是 fp32，若 Q、K 是 bf16，逐元素乘法会发生类型提升（见 5.8）。

**staff 的布局与教科书不同，但等价**

| | 输入如何配对 | 输出布局 |
|---|---|---|
| 教科书 | 相邻 $(2j,2j+1)$ | 同样交错：$[y_0,y_1,y_2,y_3,\dots]$ |
| staff | 相邻 $(2j,2j+1)$（`(half_d xy)`，xy 变化最快） | **前后半**：$[y_0,y_2,y_4,\dots\ |\ y_1,y_3,y_5,\dots]$ |

staff 输出相当于对教科书输出做一个固定置换 $P$（偶数位在前，奇数位在后）。**Q 和 K 经过同一个置换**，而点积对同时置换两个向量的分量是不变的：
$(PQ')\cdot(PK')=Q'\cdot K'$，所以注意力分数与教科书完全一致。V 不做 RoPE，不受影响。

> 前提：RoPE 之后，Q、K 的最后一维只参与 $QK^\top$ 的点积。任何依赖具体通道顺序的后续操作都会破坏这个等价性。

#### 5.6 实现

- 先按 5.2 写**教科书版**（复数或逐对公式，交错输出），再写 **staff 版**（前后半输出）。
- 接口：`forward(x, pos_ids=None)`。`pos_ids` 为 None 时取缓存的前 `seq_len` 个位置；传入时按位置索引，支持任意位置。
- 在注意力里的位置：Q、K 拆头之后、计算分数之前，$d=d_k$。

#### 5.7 对拍

| 检查 | 方法 | 预期 |
|---|---|---|
| 缓存一致 | 比较 `_freq_cis_cache` | `allclose` |
| 与 staff 一致 | 同输入同位置，fp32 | `allclose`（误差 < 1e-6） |
| 教科书版与 staff 版 | 对教科书输出取置换 `perm=[0,2,4,…,1,3,5,…]` 后与 staff 输出比较 | `allclose`；**不做置换则不相等** |
| 范数不变 | 比较 $\|y\|$ 与 $\|x\|$（每个头每个位置） | 相等 |
| 相对位置性质 | 把**同一个** $q$ 与**同一个** $k$ 复制到所有位置，得分数矩阵 $S=Q'K'^\top$ | $S[m,n]$ 与 $S[m+s,n+s]$ 相等，$S$ 沿对角线为常数 |
| 位置 0 | $m=0$ 时旋转角为 0 | 输出等于输入（置换后） |
| 手写反向 | float64 下，逆旋转与 autograd 梯度比较 | `allclose` |
| 任意位置路径 | 传入 `pos_ids` 与默认路径比较 | 一致 |

#### 5.8 与 Part 1 的关联：混合精度（3.5）

- Q、K 是线性层输出，autocast 下是 bf16；cos/sin 缓存是 fp32，**预期**逐元素乘法会提升为 fp32，因此 RoPE 的输出可能是 fp32，下一步的注意力 matmul 再被 autocast 转回 bf16。
- 这是推测，**用 forward hook 实测**后再填 3.5 的 dtype 表，不要预设答案。

#### 5.9 代价与 kernel 归属

**FLOPs**：Q、K 共 $2\cdot N\cdot d$ 个元素，每元素约 3 FLOPs，每层约 $6Nd$。small、B=4、ctx=512 时每层约 9.4 MFLOP，不到同层 matmul（约 38.7 GFLOP）的万分之三，可忽略。

**访存**：未融合的实现把一次旋转拆成许多小 kernel，每个都要完整读写一遍张量：

| 步骤 | 数量级 |
|---|---|
| 两个乘法 + 一个减法（$y_1$） | 3 个 elementwise |
| 两个乘法 + 一个加法（$y_2$） | 3 个 elementwise |
| `concat` | 1 个拷贝 |
| 上游 `rearrange` 产生的非连续视图 | 可能带来额外拷贝或跨步访问 |

每次调用至少 7 个 kernel，每层调用 2 次（Q 和 K），反向会更多。

**量级估算**（small，B=4，ctx=512，fp32）：单个 kernel 搬运约 9 MB，按 769 GB/s 约 12 µs；7 个 kernel × 24 次调用 ≈ 2 ms，大约占 forward 预期时间（约 16 ms）的一成，另有 kernel 发射开销。这是一个可在 nsys 里验证的预测，`torch.compile`（3.7）会把它们融合。

**回填第 12 章对照表**：RoPE 的 kernel 名、调用次数、占前向时间的比例。

- 注意两种rearrange方法结果

In [6]:
d = 100
x = torch.arange(d)
print(x)
# x1,x2 = rearrange(x,"... (xy half_d)->xy ... half_d",xy=2).unbind(0)
x1,x2 = rearrange(x,"... (half_d xy)->xy ... half_d",xy=2).unbind(0)
print(x1)

result = einx.id("... x_half, ... x_half -> ... (x_half (1 + 1))", x1, x2).contiguous()
#result = torch.concat((x1, x2), dim=-1)
print(result)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71,
        72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89,
        90, 91, 92, 93, 94, 95, 96, 97, 98, 99])
tensor([ 0,  2,  4,  6,  8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34,
        36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70,
        72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98])
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71,
      

In [ ]:
class  RotaryEmbedding(nn.Module):
    def __init__(
        self,
        d_k:int,
        context:int,# N
        theta:float = 10000.0,
        ):
        self.d_k = d_k
        self.ctx = context
        self.theta = theta
        self.register_buffer(
            "_freq_cis_cache", RotaryEmbedding._init_cache(context_length, dim, theta), persistent=False
        )
        self._freq_cis_cache: Float[Tensor, "2 context_length half_dim"]

    @staticmethod
    def _init_cache(context: int, d_k: int, theta: float) -> Float[Tensor, " 2 context_length half_dim"]:
        assert dim % 2 == 0

        i = torch.arange(0, dim, 2) / dim
        freqs = torch.tensor(theta) ** -i
        m = torch.arange(context_length)

        freqs = einsum(m, freqs, "m, f -> m f")

        cos, sin = torch.cos(freqs), torch.sin(freqs)
        return torch.stack((cos, sin))

    def forward(self,x):
        # no posi in
        x1, x2 = rearrange(x, "... (half_d xy) -> xy ... half_d", xy=2).unbind(0)
        seq_len = x.size(-2) # only need first seq_len
        cos, sin = self._freq_cis_cache[:, :seq_len, :].unbind(0)

        # 2D rotation matrix applied to pairs in x
        x1_rot = cos * x1 - sin * x2
        x2_rot = sin * x1 + cos * x2
        # result = einx.id("... x_half, ... x_half -> ... (x_half (1 + 1))", x1_rot, x2_rot).contiguous()
        result = torch.concat((x1_rot, x2_rot), dim=-1)
        return result


        

### 6. 因果多头自注意力

**符号**：$B$ batch，$T$ 序列长度，$d$ 模型宽度，$h$ 头数，$d_k=d/h$，$N=B\cdot T$。
单头时 $Q,K,V\in\mathbb{R}^{T\times d_k}$；缩放因子 $s=1/\sqrt{d_k}$。

#### 6.1 单头缩放点积注意力

$$
S=s\,QK^\top,\qquad P=\mathrm{softmax}(S+M),\qquad O=PV
$$

- $S\in\mathbb{R}^{T\times T}$：第 $(i,j)$ 项是"位置 $i$ 的 query 对位置 $j$ 的 key"的匹配分数。
- softmax 对**每一行**（沿 key 维）做，所以 $P$ 的每行是一个概率分布。
- $M$ 是因果 mask（见 6.4）。输出 $o_i=\sum_j p_{ij}v_j$：每个位置是 value 的加权平均。

#### 6.2 为什么除以 $\sqrt{d_k}$

设 $q,k$ 的各分量独立、均值 0、方差 1，则

$$
\mathrm{Var}(q\cdot k)=\sum_{a=1}^{d_k}\mathrm{Var}(q_ak_a)=d_k,\qquad
\mathrm{Var}(s\,q\cdot k)=1
$$

- 不缩放时分数的标准差是 $\sqrt{d_k}$（$d_k=64$ 时为 8），softmax 输入的差距很大，输出接近 one-hot。
- softmax 的雅可比是 $\mathrm{diag}(p)-pp^\top$（见 6.3），one-hot 时趋于 0，**梯度消失**。
- 这个假设在本作业的初始化下近似成立：RMSNorm 输出每个分量的均方为 1，$W_q,W_k$ 的元素方差约 $1/d$，所以 $q,k$ 分量方差约 1（截断正态使它略小于 1）。RoPE 保持范数，不改变这一点。
- 可数值检验：初始化后测 $\mathrm{Var}(S)$，缩放后应略小于 1，不缩放则约为 $d_k$。

#### 6.3 softmax：稳定写法与雅可比

**定义**：$p_i=e^{x_i}/\sum_j e^{x_j}$。

**平移不变性**：对任意常数 $c$，$\mathrm{softmax}(x)=\mathrm{softmax}(x-c)$。取 $c=\max_j x_j$：
- 指数的自变量都 $\le0$，**不会上溢**；
- 最大项恰为 $e^0=1$，分母 $\ge1$，**不会除零**。

| 类型 | $e^x$ 上溢的阈值 | 说明 |
|---|---|---|
| fp32 | $x\gtrsim 88.7$ | 平常较少触发 |
| fp16 | $x\gtrsim 11.1$ | 很容易触发 |
| bf16 | 同 fp32 的范围 | 不上溢，但尾数短，求和误差大 |

**雅可比**：

$$
\frac{\partial p_i}{\partial x_j}=p_i(\delta_{ij}-p_j)
\;\Rightarrow\; J=\mathrm{diag}(p)-pp^\top
$$

- $J$ 对称，且 $J\mathbf{1}=0$（对应平移不变性），所以秩至多 $n-1$。
- **不要显式构造 $J$**（$n\times n$）。反向用向量形式：

$$
\boxed{\delta x=p\odot\big(\delta p-\langle\delta p,\,p\rangle\big)}
$$

只需要 $p$ 和一次行内点积，复杂度 $O(n)$。

#### 6.4 因果 mask

**构造**：$M_{ij}=0$（$i\ge j$），$-\infty$（$i<j$）。源码里用布尔矩阵 `causal_mask = (qi >= kj)`（True 表示保留），再用 `torch.where(mask, scores, -inf)`。

**$-\infty$ 的作用**：$e^{-\infty}=0$，被屏蔽位置的概率**恰好为 0**，不是"很小"。反向时 $\delta s=p\odot(\dots)$，这些位置的梯度也恰好为 0。

**两个边界**：
- 每行至少保留对角线，所以不会出现"整行都是 $-\infty$"。若出现，softmax 会算出 $0/0$，得到 NaN。
- 第 0 行只能看自己，$p_{00}=1$。

**系统含义**：因果 mask 是下三角，上三角的一半是无效计算。vanilla 实现仍然算完整的 $T\times T$ 再屏蔽；FlashAttention 会跳过完全被屏蔽的分块（Part 2 的 M4）。

#### 6.5 注意力的反向（单头）

已知 $\delta O$，逐步反推：

$$
\delta V=P^\top\delta O,\qquad
\delta P=\delta O\,V^\top,\qquad
\delta S=P\odot\big(\delta P-D\mathbf{1}^\top\big),
$$
$$
\delta Q=s\,\delta S\,K,\qquad
\delta K=s\,\delta S^\top Q
$$

其中 $D_i=\sum_j P_{ij}\,\delta P_{ij}$。**化简**：$\delta P_{ij}=\delta o_i\cdot v_j$，所以

$$
D_i=\delta o_i\cdot\sum_jP_{ij}v_j=\delta o_i\cdot o_i
\;\Rightarrow\; D=\mathrm{rowsum}(\delta O\odot O)
$$

- 这个技巧**不需要 $P$ 的行内归约**，只需要 $O$ 和 $\delta O$，是 FlashAttention 反向的核心（Part 2 会用到）。
- backward 有 4 个 matmul（$\delta V,\delta P,\delta Q,\delta K$），forward 有 2 个，所以 backward 的 matmul FLOPs 是 forward 的 2 倍，与 Step 2 的 1:2 一致。
- $P$ 需要在前向保存，这是**显存的主要来源**（见 6.11）。

#### 6.6 多头

1. 投影：$Q=XW_Q^\top$，$K=XW_K^\top$，$V=XW_V^\top$，各是 $d\to d$。
2. 拆头：把最后一维 $d$ 切成 $h$ 段，每段 $d_k$：`(B,T,d) → (B,h,T,d_k)`。
3. 各头独立做 6.1。
4. 拼接：`(B,h,T,d_k) → (B,T,h·d_k)`。
5. 输出投影

In [8]:
def softmax(x, dim=-1):
    rescaled_input = x - torch.max(x, dim=dim, keepdim=True)[0]
    exponentiated_rescaled_input = torch.exp(rescaled_input)
    return exponentiated_rescaled_input / torch.sum(exponentiated_rescaled_input, dim=dim, keepdim=True)

- scaled_dot_product_attention
> 该函数无需训练参数，直接定义即可

In [9]:
def scaled_dot_product_attention(
    Q: Float[Tensor, " ... queries d_k"],
    K: Float[Tensor, " ... keys    d_k"],
    V: Float[Tensor, " ... keys    d_v"],
    mask: Bool[Tensor, " ... queries keys"] | None = None,
)->Float[Tensor,"... d_v"]:
    d_k = Q.shape[-1]
    x = einsum(Q,K,"... queries d_k, ... keys d_k -> ... keys")/math.sqrt(d_k)
    if mask is not None:
            x = torch.where(mask, x, float("-inf"))
    x = softmax(x)
    return einsum(x,V,"... queries keys, ... keys d_v-> ... quesries d_v")

- CausalMultiHeadSelfAttention(nn.Module):
1. `rearrange(X,"... seq (head_nums d_k)-> ... head_nums seq d_k",head_nums=self.head_nums)`

    >这里重排即自动解耦（attn计算定义时只要求了最后两维[s,d_k]）

2. 注意mask构造方法，插入维度使得可以广播

In [17]:
# mask
sequence_length = 100
batch_dims = torch.zeros(3,1)
iota = torch.arange(sequence_length, device=x.device)
qi = rearrange(iota, "query -> query 1")
kj = rearrange(iota, "key   -> 1   key")
causal_mask = qi >= kj  # (query, key)
print(causal_mask)
print(causal_mask.shape)
causal_mask = causal_mask.__getitem__((None,) * len(batch_dims) + (...,))  # Add appropriate leading dimensions
print(causal_mask)
print(causal_mask.shape)

tensor([[ True, False, False,  ..., False, False, False],
        [ True,  True, False,  ..., False, False, False],
        [ True,  True,  True,  ..., False, False, False],
        ...,
        [ True,  True,  True,  ...,  True, False, False],
        [ True,  True,  True,  ...,  True,  True, False],
        [ True,  True,  True,  ...,  True,  True,  True]])
torch.Size([100, 100])
tensor([[[[[ True, False, False,  ..., False, False, False],
           [ True,  True, False,  ..., False, False, False],
           [ True,  True,  True,  ..., False, False, False],
           ...,
           [ True,  True,  True,  ...,  True, False, False],
           [ True,  True,  True,  ...,  True,  True, False],
           [ True,  True,  True,  ...,  True,  True,  True]]]]])
torch.Size([1, 1, 1, 100, 100])


In [19]:
class CausalMultiHeadSelfAttention(nn.Module):
    def __init__(
            self,
            d_model:int,
            head_nums:int,
            positional_encoder: RotaryEmbedding | None = None,
    ):
        super.__init__()
        assert d_model%head_nums==0
        if positional_encoder==None:
            warnings.warn("No positional encoder provided", stacklevel=2)
        self.d_k = d_model//head_nums
        self.d_model = d_model
        self.head_nums = head_nums
        self.d_v = self.d_k

        self.proj_q = Linear(self.d_model,self.head_nums*self.d_k)
        self.proj_k = Linear(self.d_model,self.head_nums*self.d_k)
        self.proj_v = Linear(self.d_model,self.head_nums*self.d_v)

        self.proj_out = Linear(self.head_nums*self.d_v,self.d_model)
        self.positinal_encoder = positional_encoder

    def forward(
        x:Float[Tensor,"... d_model"]
    )->Float[Tensor,"... d_model"]:
        *batch_dims, sequence_length, d_model = x.size()
        assert d_model == self.d_model

        # proj
        Q = self.proj_q(x)
        K = self.proj_k(x)
        V = self.proj_v(x)

        # split
        Q,K,V = (
            rearrange(X,"... seq (head_nums d_k)-> ... head_nums seq d_k",head_nums=self.head_nums) 
            for X in(Q,K,V)
            )# [b,s,h*d_k] -> [b,h,s,d_k]

        # RoPE
        if self.positional_encoder is not None:
            Q = self.positional_encoder(Q)
            K = self.positional_encoder(K)

        # mask
        iota = torch.arange(sequence_length, device=x.device)
        qi = rearrange(iota, "query -> query 1")
        kj = rearrange(iota, "key   -> 1   key")
        causal_mask = qi >= kj  # (query, key)
        causal_mask = causal_mask.__getitem__((None,) * len(batch_dims) + (...,))  # Add appropriate leading dimensions

        # attn
        attn = scaled_dot_product_attention(Q,K,V,causal_mask) #[b,h,s,d_v]

        # concat&proj
        y = rearrange(attn,"... h s d_v -> ... s (h d_v)") #[b,h,s,d_v]->[b,s,h*d_v]
        y = self.proj_out(y) #[b,s,s*d_v]->[b,s,d_model]
        return y


### 7. SwiGLU 前馈网络

**符号**：$N=B\cdot T$ 为 token 总数，$d$ 为模型宽度，$d_{ff}$ 为内层宽度。逐 token 独立计算，没有 token 间的交互。
**约定**：沿用 staff 的行向量写法，`Linear` 是 $y=xW^\top$，权重形状为 `(d_out, d_in)`。

#### 7.1 结构

$$
a=xW_1^\top,\quad b=xW_3^\top,\quad u=\mathrm{SiLU}(a),\quad g=u\odot b,\quad y=gW_2^\top
$$

| 矩阵 | staff 参数名 | 形状 | 角色 |
|---|---|---|---|
| $W_1$ | `w1.weight` | $(d_{ff},d)$ | **门**分支，输出经过 SiLU |
| $W_3$ | `w3.weight` | $(d_{ff},d)$ | **值**分支，不经过非线性 |
| $W_2$ | `w2.weight` | $(d,d_{ff})$ | 降维，把 $d_{ff}$ 投回 $d$ |

注意编号不是按顺序的：门是 $W_1$，值是 $W_3$。源码 `w2(silu(w1(x)) * w3(x))` 里的 `silu` 只作用在 `w1` 上，写反会导致 `load_state_dict` 通过但结果不一致。

#### 7.2 SiLU

$$
\mathrm{SiLU}(x)=x\,\sigma(x),\qquad \sigma(x)=\frac{1}{1+e^{-x}}
$$

**导数**：利用 $\sigma'=\sigma(1-\sigma)$，

$$
\mathrm{SiLU}'(x)=\sigma(x)+x\,\sigma(x)\big(1-\sigma(x)\big)=\sigma(x)\big(1+x(1-\sigma(x))\big)
$$

**性质**（均可数值验证）：
- $\mathrm{SiLU}(0)=0$，$\mathrm{SiLU}'(0)=0.5$；
- $x\to+\infty$ 时趋近 $x$；$x\to-\infty$ 时趋近 0；
- **非单调**：在 $x\approx-1.28$ 处有最小值约 $-0.28$，所以允许小幅负输出；
- 导数最大值略大于 1（约 1.1，在 $x\approx2.4$ 附近）；
- 光滑，处处可导，没有 ReLU 在 0 处的拐点。它也叫 Swish-1（$x\sigma(\beta x)$ 取 $\beta=1$）。

#### 7.3 门控机制

**GLU 的想法**：用一个分支控制另一个分支的通过量。

$$
\mathrm{GLU}(x)=\sigma(xW)\odot(xV)
$$

把门里的 $\sigma$ 换成别的函数，得到一族变体（ReGLU、GEGLU、SwiGLU 等，见 Shazeer 2020《GLU Variants Improve Transformer》）。SwiGLU 取 SiLU 作门，再加一个降维矩阵 $W_2$。

**直觉**：
- 标准 MLP 是 $W_2\,\phi(W_1x)$，非线性只对每个通道**单独**作用；
- 门控是两个**线性投影的逐元素乘积**，引入了乘性交互：同一个 token 在每个通道上被"调制"了多少，由 $W_1$ 决定，内容由 $W_3$ 决定；
- 经验上，在相同参数量或相同计算量下，门控版本的效果一般更好，所以被 LLaMA 一类模型采用。这是实验结论，不是推导出来的。

#### 7.4 反向传播

给定 $\delta y$，按计算图倒推：

$$
\delta g=\delta y\,W_2,\qquad \delta W_2=\delta y^\top g
$$
$$
\delta b=\delta g\odot u,\qquad \delta u=\delta g\odot b,\qquad \delta a=\delta u\odot \mathrm{SiLU}'(a)
$$
$$
\delta x=\delta a\,W_1+\delta b\,W_3,\qquad \delta W_1=\delta a^\top x,\qquad \delta W_3=\delta b^\top x
$$

（$\delta W$ 的矩阵乘法在 $N$ 个 token 上求和，$\delta y$ 等按 `(N, ·)` 展平。）

要点：
- 输入 $x$ **同时流入两个分支**，所以 $\delta x$ 是两条路径的**梯度之和**。
- backward 有 **6 个 matmul**（每个线性层 2 个），forward 有 3 个，所以 backward 的 matmul FLOPs 是 forward 的 2 倍，与 Step 2 一致。
- 反向需要保存：$x$、$a$、$b$、$u$、$g$（以及手写 SiLU 里的 $\sigma(a)$），这就是 7.9 里激活显存的来源。

#### 7.5 参数量与 $d_{ff}$ 的选择

每层 FFN 参数：$3\,d\,d_{ff}$（staff 无 bias）。

| FFN 类型 | 矩阵数 | $d_{ff}$ | FFN 参数 | 整层参数（含注意力 $4d^2$） |
|---|---|---|---|---|
| GPT-2 的 GELU MLP | 2 | $4d$ | $8d^2$ | $12d^2$ |
| SwiGLU，**本作业** | 3 | $4d$ | $12d^2$ | $16d^2$（GPT-2 的 1.33 倍） |
| SwiGLU，LLaMA 惯例 | 3 | $\frac83d$（取 64 的倍数） | $8d^2$ | $12d^2$（与 GPT-2 持平） |

- **本作业的 $d_{ff}$ 取值**：small 3072、medium 4096、large 5120、xl 6400、2.7B 10240，全是 $4d$。
- **这就是 Step 2 里"实测参数量大于 GPT-2 名义规模"的原因**：medium 约 0.42B（名义 0.35B），large 约 0.97B（名义 0.76B）。
- $d_{ff}$ 取 64 的倍数是为了对齐 tensor core 的分块，对 matmul 效率有影响。
- 代码里 `d_ff` 是**独立的超参数**，没有强制等于 $4d$。

#### 7.6 实现

- 先手写 `silu`（`x * sigmoid(x)`），再对照 `F.silu`。
- 三个 `Linear`，参数名 `w1`、`w2`、`w3`，初始化沿用第 3 章的 Xavier 型截断正态（$\sigma=\sqrt{2/(d_{in}+d_{out})}$）。
- `forward`：`w2(silu(w1(x)) * w3(x))`。
- **两种 `silu` 的系统差异**（后面做对比实验，见 7.9）：

| | 手写 `x*sigmoid(x)` | `F.silu` |
|---|---|---|
| kernel 数（forward） | 2（sigmoid、mul） | 1 |
| 访存（单位：一个 $N\times d_{ff}$ 张量的读写次数） | 5 | 2 |
| 保存给反向的张量 | $a$、$\sigma(a)$ | $a$（反向时重算 $\sigma$） |

#### 7.7 对拍

| 检查 | 方法 | 预期 |
|---|---|---|
| SiLU 前向 | 与 `F.silu` 比较 | `allclose` |
| SiLU 导数 | 7.2 的公式与 autograd 比较（float64） | `allclose` |
| SiLU 的性质 | 在网格上找最小值 | 位置约 $-1.28$，值约 $-0.28$ |
| 参数量 | `sum(p.numel())` | 等于 $3\,d\,d_{ff}$ |
| 与 staff 一致 | 同权重同输入，fp32 | `allclose`（误差 < 1e-5） |
| 门/值是否写反 | 交换 `w1`、`w3` 的权重，输出应**明显改变** | 与原输出不同，说明二者角色不对称 |
| 手写反向 | 按 7.4 在 float64 下算 $\delta x,\delta W_{1,2,3}$ | 与 autograd `allclose` |
| 梯度检验 | `torch.autograd.gradcheck`（小尺寸，float64） | 通过 |
| 逐 token 独立 | 打乱序列中 token 顺序，输出应同样被打乱 | 成立（FFN 没有 token 间交互） |
| 梯度累加 | 检查 $\delta x$ 是两条分支贡献之和 | 与分别反传再相加一致 |

#### 7.8 与 Part 1 的关联

- **3.5 混合精度**：`Linear`（einsum 底层是批量矩阵乘）在 autocast 下**预期**会变成 bf16；`silu` 与逐元素乘不在 autocast 的类型转换名单里，**预期**沿用输入的 dtype，即 bf16。这是预期，用 forward hook 实测后再填表。
- **3.7 `torch.compile`**：SwiGLU 中间的"sigmoid、mul、mul"是一串逐元素运算，是编译融合的直接对象。
- **一个可做的小实验**：把手写 `silu` 换成 `F.silu`，对比 forward 时间与峰值显存。预期显存少一份 $N\times d_{ff}$，时间略降。

#### 7.9 代价与 kernel 归属

**(1) FLOPs（每层 forward）**

$$
\underbrace{2N\,d\,d_{ff}}_{W_1}+\underbrace{2N\,d\,d_{ff}}_{W_3}+\underbrace{2N\,d\,d_{ff}}_{W_2}=6N\,d\,d_{ff}=24N\,d^2\ \ (d_{ff}=4d)
$$

与注意力合起来，每层 forward 为 $32Nd^2+4NTd$（线性层 $8Nd^2$ 的注意力投影加上注意力 matmul），所以 FFN 占比：

$$
\text{FFN 占比}=\frac{24Nd^2}{32Nd^2+4NTd}=\frac{6d}{8d+T}
$$

| 模型 | $T=128$ | $T=512$ | $T=1024$ |
|---|---|---|---|
| small（$d=768$） | 73.5% | 69.2% | 64.3% |
| large（$d=1280$） | 74.1% | 71.4% | 68.2% |

结论：FFN 是每层 FLOPs 的约三分之二到四分之三，但**不是全部**，注意力的投影和 matmul 合计仍占四分之一以上；T 越大，注意力的占比越高。

**示例**（small，$B=4,T=512$）：每层 FFN 约 29.0 GFLOP，按 $P_{fp32}=33.4$ TFLOPs/s 约 0.87 ms，12 层约 10.4 ms。

**(2) 访存：SwiGLU 中间的逐元素运算**

forward 的三个 elementwise kernel（手写 SiLU）：

| kernel | 读 | 写 | 读写张量数 |
|---|---|---|---|
| `sigmoid(a)` | $a$ | $\sigma$ | 2 |
| `a * sigmoid` | $a,\sigma$ | $u$ | 3 |
| `u * b` | $u,b$ | $g$ | 3 |

合计 8 个 $N\times d_{ff}$ 张量的读写。small 下一个这样的张量是 $2048\times3072\times4\,\text{B}\approx25.2$ MB：
- 每层约 $8\times25.2=201$ MB，按 769 GB/s 约 0.26 ms；12 层约 3.1 ms；
- 对比同层 FFN 的 matmul（约 0.87 ms），**逐元素部分约为其 30%**；
- 融合成一个 kernel（读 $a,b$，写 $g$）只需 3 个张量，节省约 60%（`F.silu` 版本是 5 个，节省约 38%）。

这是可在 nsys 里验证的预测。

**(3) 激活显存：每层保存 5 份 $N\times d_{ff}$**

autograd 为反向保存的张量（手写 SiLU）：

| 张量 | 谁保存 |
|---|---|
| $a=xW_1^\top$ | `x * sigmoid(x)` 的乘法 |
| $\sigma(a)$ | 同上，且 `sigmoid` 自己的反向也要它 |
| $u$ | `u * b` 的乘法 |
| $b$ | 同上 |
| $g$ | $W_2$ 的权重梯度 |

此外还有输入 $x$（$N\times d$，两个分支共用）。所以每层约 $5\,N\,d_{ff}\times$ 字节数，用 `F.silu` 则是 4 份。

fp32，$B=4,T=512$（$N=2048$）：

| 模型 | $d_{ff}$ | 每层 (MB) | 总计 (GiB) |
|---|---|---|---|
| small | 3072 | 126 | 1.4 |
| medium | 4096 | 168 | 3.8 |
| large | 5120 | 210 | 7.0 |

这些只是 FFN 的部分，且是预期值，实际要用 memory snapshot 确认（einsum 内部是否产生额外拷贝会影响份数）。

**(4) 与注意力的激活对比**

注意力分数矩阵每层保存 $c$ 份 $B\,h\,T^2$（$c$ 约 1 到 3，取决于 `softmax` 的实现），FFN 保存 $5Nd_{ff}=20Nd$ 个元素，二者之比：

$$
\frac{\text{注意力分数}}{\text{FFN}}=\frac{c\,B\,h\,T^2}{20\,B\,T\,d}=\frac{c\,T}{20\,d_k}
$$

- $d_k=64$ 时为 $cT/1280$，**与模型宽度无关**；
- $T$ 较小时 FFN 的激活更大，$T$ 超过约 $1280/c$ 之后注意力的 $T^2$ 项接管。

**(5) 对 3.2 的提示：OOM 边界要靠实测确定**

之前只算了静态显存。加上激活（只按 FFN 与注意力分数粗估，**还没算** RMSNorm、RoPE、Q/K/V 等 $N\times d$ 的张量和 logits）：

| 配置 | 静态 (GiB) | FFN 激活 | 注意力分数（$c=2$） | 小计 |
|---|---|---|---|---|
| medium，$T=512$ | 6.3 | 3.8 | 3.0 | 约 13.1，加上其他已贴近 16GB |
| medium，$T=1024$ | 6.3 | 7.5 | 12.0 | 大概率 OOM |
| large，$T=128$（不含 optimizer） | 7.2（权重+梯度） | 1.8 | 0.4 | 约 9.4，可行 |
| large，$T=256$（不含 optimizer） | 7.2 | 3.5 | 1.4 | 约 12.1，加上其他贴近上限 |

做法：
- medium 的完整训练步，**从 ctx=128、256 开始逐步加大**，找到 OOM 的边界。这个边界本身就是 3.2 的状态列。
- large 的 forward+backward 从 ctx=128 开始；完整训练步（含 Adam）在本机必然 OOM。
- 在 notebook 里写一个函数 `est_act_bytes(model, B, T, silu_impl, c)`，输出预估显存，再与 `max_memory_allocated` 对比，误差本身值得分析。

**(6) 回填第 12 章对照表**

| 代码位置 | 预期 kernel | 实测 kernel 名 | 调用次数/层 | 占 forward 比例 |
|---|---|---|---|---|
| `w1(x)`、`w3(x)` | GEMM × 2 | | | |
| `sigmoid` | elementwise | | | |
| `x * sigmoid(x)` | elementwise | | | |
| `silu(...) * w3(x)` | elementwise | | | |
| `w2(...)` | GEMM | | | |

In [20]:
def silu(x: torch.Tensor):
    return x * torch.sigmoid(x)

### 8. Transformer Block
- 数学：pre-norm 与 post-norm 的区别，残差连接对梯度的作用
- 画出一个 block 的数据流图，标出每个算子的类型（计算密集 / 访存密集）
- 对拍

### 9. 整体拼装
- 组装：embedding → L 个 block → ln_final → lm_head
- 参数量公式推导，并与 `sum(p.numel())` 对照（接 Step 2）
- 为什么 `lm_head` 不绑定，以及绑定会改变什么

### 10. 与 staff 的整体验证
- 对拍流程：
  1. 构造 staff 模型，随机初始化
  2. 构造 mini 模型，`load_state_dict(staff.state_dict(), strict=True)`
  3. 同一批 token，比较 logits（`max abs diff`，fp32 下应在 1e-5 量级）
  4. 同一个 loss，比较所有参数的梯度
- 失败时的定位方法：用 forward hook 逐模块比较，二分到出错层
- 在 meta 设备上核对参数量

### 11. 代价核算
- 每个模块一行：FLOPs 公式、激活显存、计算密集或访存密集
- 汇总成一个训练步的 FLOPs 拆分图：线性层、lm_head、注意力三项各占多少
- 结论：为什么 ctx 增大时注意力占比上升
- 这一章把 Step 2 的手算与模型结构对应起来

### 12. 代码 → kernel 对照表与 NVTX 标注点
- 先留空表，等 nsys 结果出来后回填：

| 代码位置 | 预期 kernel 类型 | nsys 中的实际 kernel 名 | 占比 |
|---|---|---|---|
| `q_proj` 的 einsum | GEMM |  |  |
| RoPE | elementwise |  |  |
| softmax | reduce + elementwise |  |  |
| ... |  |  |  |

- 标注计划：forward、backward、optimizer，以及 attention 内部的 scores、softmax、AV 三段

### 13. 附录
- 推导：softmax + cross-entropy 的梯度、RMSNorm 反向、RoPE 的相对位置性质
- 自测题（每章 2 到 3 道），例如：
  - 为什么 backward 约为 forward 的 2 倍？
  - 哪些算子在 bf16 autocast 下会被 cast，哪些不会？
  - 为什么 T 加倍时注意力显存变为 4 倍？
- 参考：原论文（Attention Is All You Need、RoPE、RMSNorm、GLU Variants）

## 使用建议

- **做完第 3、5、6 章再决定要不要继续**：这三章最能帮你看清 matmul、softmax、elementwise 在哪里，是 nsys 归因真正需要的。
- **先写 `MiniTransformerLM` 的最小版本（第 9、10 章先通），再回头补推导**。先通过对拍，后面的数学注解才有可验证的落点。
- **做完后把核心类抽到 `cs336_systems/mini_model.py`**，让 `benchmark.py` 可以通过一个开关在 staff 模型和 mini 模型之间切换，这本身就是一个很好的交叉验证。
- Jupyter 在远程服务器上用端口转发：服务器端 `uv run jupyter lab --no-browser --port 8888`，本地 `ssh -L 8888:localhost:8888 <user>@c220`。